# Train a model using Autogluon

After preparing the data in the previous step, we read the data, stored in gold, to train a model.

In a real scenario, we would have corrected data for batch effects, which are clearly shown in the previous notebook.

Here we train a simple model to predict healthy/disease, then log it into mlflow.


## Install Libraries

We use autogluon, a popular autoML tool from Amazon.

In [0]:
%pip install -q autogluon
import autogluon

## Read data

The data has already been prepared in gold, we just need to read it.

In [0]:
!pip install networkx

In [0]:
 dbutils.library.restartPython()

In [0]:
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.metrics import roc_auc_score, confusion_matrix
import mlflow
import mlflow.sklearn

# Step 0 - Read sample metadata
sample_meta = pd.read_parquet("/Volumes/gold/methylation/methylation_features/sample_metadata.parquet")

sample_meta.head()


In [0]:
sample_meta.sample_group.map({"disease": 1, "healthy":0})

In [0]:
sample_meta["sample_group"] = sample_meta.sample_group.map({"disease": 1, "healthy":0})

In [0]:
# Step 1: Load wide-format methylation matrix from Parquet
df = pd.read_parquet("/Volumes/gold/methylation/methylation_features/methylation_beta_001.parquet")
df.head()


In [0]:
#!pip install neurocombat_sklearn

In [0]:

# Step 2: Join sample_group info
df["sample_group"] = sample_meta.loc[df.index, "sample_group"].values

# Step 3: Drop missing target values
df = df.dropna(subset=["sample_group"])
label_col = "sample_group"
X = df.drop(columns=[label_col])
y = df[label_col]

# Step 4: Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

train_data = X_train.copy()
train_data[label_col] = y_train
test_data = X_test.copy()
test_data[label_col] = y_test

# Step 5: Start MLflow tracking

In [0]:
print(mlflow.get_tracking_uri())

In [0]:
# manual fix to get mlflow to work in a Serverless compute
# https://community.databricks.com/t5/machine-learning/using-datbricks-connect-with-serverless-compute-and-mlflow/td-p/97590

import mlflow
import databricks.connect as db_connect
import mlflow.tracking._model_registry.utils

# Workaround to set the registry URI manually
mlflow.tracking._model_registry.utils._get_registry_uri_from_spark_session = lambda: "databricks-uc"

mlflow.login() # This prints an INFO-log: Login successful!
# mlflow.set_model_uri("databricks")
spark_ctx = db_connect.DatabricksSession.builder.serverless(True).getOrCreate()
#train_and_log_ml_model(spark_ctx)

In [0]:

mlflow.set_tracking_uri("databricks")
with mlflow.start_run(run_name="AutoGluon_Methylation_Classifier"):
    
    # Step 6: Train with AutoGluon
    predictor = TabularPredictor(label=label_col, eval_metric="roc_auc", verbosity=2).fit(train_data, time_limit=1800)

    # Step 7: Evaluate
    y_pred_proba = predictor.predict_proba(test_data)["healthy"]  # get probability of positive class
    y_pred = predictor.predict(test_data)
    y_true = test_data[label_col]

    # Compute metrics
    auroc = roc_auc_score(y_true, y_pred_proba)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=predictor.class_labels).ravel()
    sensitivity = tp / (tp + fn)  # recall
    specificity = tn / (tn + fp)  # specificity

    # Step 8: Log metrics
    mlflow.log_metric("AUROC", auroc)
    mlflow.log_metric("Sensitivity", sensitivity)
    mlflow.log_metric("Specificity", specificity)

    # Log model
    # Save the AutoGluon model locally
    #predictor.save("autogluon_model")

    # Log the folder as MLflow artifacts (NOT as a registered model)
    mlflow.log_artifacts("autogluon_model", artifact_path="model")



    print(f"✅ AUROC: {auroc:.3f}, Sensitivity: {sensitivity:.3f}, Specificity: {specificity:.3f}")


In [0]:
specificity